BIBLIOTECAS

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import RandomizedSearchCV

LEITURA DOS ARQUIVOS DESCRITORES DE ETNIAS E LÍNGUAS - CLASSIFICAÇÕES PRÉVIAS E ATUAIS

In [ ]:
caminho_arquivo = 'BDD_etnias_20250505.xlsx';
df_linguas_coleta = pd.read_excel(caminho_arquivo);

caminho_arquivo = 'BDD_linguas_20240604.xlsx';
df_linguas_coleta = pd.read_excel(caminho_arquivo);

caminho_arquivo = 'descritores.xlsx';
df_linguas_novacod = pd.read_excel(caminho_arquivo, sheet_name='etnias');
df_linguas_novacod = pd.read_excel(caminho_arquivo, sheet_name='linguas');

df_ind = pd.read_csv('df_ind_distancia.csv', sep=';', encoding='utf-8-sig')

DEFINIÇÃO DE VARIÁVEIS NUMÉRICAS E CATEGÓRICAS

In [ ]:
variaveis_numericas = ['VAL_LATITUDE','VAL_LONGITUDE','cod_setor','distancia']
variaveis_categoricas= ['txt_etnia_entrada_coleta','desc_cod_lingua_1_novacod','tipo_setor', 'CD_TI', 'sobrenome','descricao_mais_proxima']

GARANTIR QUE O DATASET DE TREINO TENHA AO MENOS UM REGISTRO DE CADA CLASSIFICAÇÃO RÓTULO

In [ ]:
# Índices com cardinalidade 1
cardinalidade_etnia = df_ind['desc_cod_etnia_1_novacod'].value_counts()
cardinalidade1_etnia = cardinalidade_etnia[cardinalidade_etnia == 1].index

# Registros com cardinalidade 1
df_cardinalidade1 = df_ind[df_ind['desc_cod_etnia_1_novacod'].isin(cardinalidade1_etnia)].copy()

# Registros com cardinalidade diferente de 1
df_cardinalidadedif1 = df_ind[~df_ind['desc_cod_etnia_1_novacod'].isin(cardinalidade1_etnia)].copy()

# Split treino e validacao_teste
X_treino, X_validacao_teste, y_treino, y_validacao_teste = train_test_split(
    df_cardinalidadedif1.drop(columns=['desc_cod_etnia_1_novacod']),  
    df_cardinalidadedif1['desc_cod_etnia_1_novacod'],
    test_size=0.4,
    stratify=df_cardinalidadedif1['desc_cod_etnia_1_novacod']
)

# Inclusão dos registros de cardinalidade 1 no dataset de treino
X_treino = pd.concat([X_treino, df_cardinalidade1.drop(columns=['desc_cod_etnia_1_novacod'])])
y_treino = pd.concat([y_treino, df_cardinalidade1['desc_cod_etnia_1_novacod']])

# Split validacao e teste
X_teste, X_validacao, y_teste, y_validacao = train_test_split(
    X_validacao_teste,  
    y_validacao_teste,
    test_size=0.5,
)

PIPELINES DE PRÉ-PROCESSAMENTO COM ORDINAL E ONE HOT ENCONDERS

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_pipeline_ordinal = Pipeline(steps=[
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

categorical_pipeline_onehot = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor_ordinal = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, variaveis_numericas),
        ('cat', categorical_pipeline_ordinal, variaveis_categoricas)
    ]
)

preprocessor_onehot = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, variaveis_numericas),
        ('cat', categorical_pipeline_onehot, variaveis_categoricas)
    ]
)

display(preprocessor_ordinal)
display(preprocessor_onehot)

PIPILINE DO RANDOM FOREST

In [ ]:
floresta = Pipeline(steps=[
    ('preprocessor', preprocessor_ordinal),
    ('classifier', RandomForestClassifier())
])

floresta

TUNING COM RANDOM SEARCH DO PIPELINE DO MODELO RANDOM FOREST 

In [ ]:
parametros = {
    'classifier__n_estimators': np.arange(50, 500, 50),
    'classifier__max_depth': [None, 10, 20, 30],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__bootstrap': [True, False],
}

random_search = RandomizedSearchCV(
    estimator=floresta, 
    param_distributions=parametros, 
    n_iter=50, 
    scoring="f1_macro",
    cv=5, 
    verbose=1, 
    n_jobs=3
)

random_search.fit(X_treino, y_treino)

print(f"Melhores parâmetros: {random_search.best_params_}")
print(f"F1 Score: {random_search.score(X_validacao, y_validacao):.4f}")